In [223]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from optuna.samplers import TPESampler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
!pip install optuna scikit-optimize
from skopt import BayesSearchCV
import optuna
from scipy.stats import randint

In [224]:
data = load_breast_cancer()
X = data.data
y = data.target

In [225]:
X.shape

(569, 30)

In [226]:
X_train, X_test, y_train, y_test =  train_test_split(X,y, random_state=42, stratify=y)

In [227]:
model = XGBClassifier(eval_metric="logloss", random_state=42)
#métrica de evaluación para modelos de clasificación que mide qué tan acertadas y seguras son sus predicciones de probabilidad.
#LogLoss evalúa el valor numérico de la probabilidad asignada (de 0 a 1). ejemplo:
# Si el modelo predice que un evento ocurrirá con 99% de certeza y resulta ser falso, se penaliza fuertemente.

# **Hiperparamteros**
**max_depth.-** La profundidad máxima que puede alcanzar cada árbol. Valores grandes hacen el modelo complejo y propenso a aprender ruido.

**n_estimators.-** La cantidad total de árboles que construirá el algoritmo,demasiados pueden provocar sobreajuste si la tasa de aprendizaje es alta.

**min_child_weight.-** Define qué tan exigente es el árbol para dividirse. Un valor alto evita que el modelo cree ramas para pocos datos (previniendo el sobreajuste), mientras que un valor bajo permite divisiones más complejas

In [228]:
search_space = { "min_child_weight": (1,10), "max_depth":(2,10), "n_estimators":(50,300)   }

**Optimizacion Bayesiana**

Creo el evaluador de pruebas (funcion objetivo)

In [229]:
def objetivo(prueba):
  mininmo_child_weight = prueba.suggest_int("min_child_weight",1,10)
  maxima_depth = prueba.suggest_int("max_depth",1,10)
  num_estimators = prueba.suggest_int("n_estimators",50,300)

  model = XGBClassifier(
      min_child_weight=mininmo_child_weight,
      max_depth= maxima_depth,
      n_estimators= num_estimators,
      eval_metric="logloss",
      random_state=42
  )

  model.fit(X_train,y_train)
  acuracy = model.score(X_test,y_test)
  return acuracy

Ejecutar la optimizacion

In [230]:
semilla_optuna=TPESampler(seed=24)
contenedor = optuna.create_study(direction="maximize", sampler=semilla_optuna) # create_study crea una instancia de estudio (experimentos/objetivos de optimizacion)

[I 2026-08-25 06:10:26,192] A new study created in memory with name: no-name-76ed7e4e-2442-4bdf-94d1-a6cf30d1ed04


In [231]:
contenedor.optimize(objetivo, n_trials=20)

[I 2026-08-25 06:10:26,269] Trial 0 finished with value: 0.972027972027972 and parameters: {'min_child_weight': 10, 'max_depth': 7, 'n_estimators': 300}. Best is trial 0 with value: 0.972027972027972.
[I 2026-08-25 06:10:26,895] Trial 1 finished with value: 0.972027972027972 and parameters: {'min_child_weight': 3, 'max_depth': 4, 'n_estimators': 235}. Best is trial 0 with value: 0.972027972027972.
[I 2026-08-25 06:10:26,928] Trial 2 finished with value: 0.972027972027972 and parameters: {'min_child_weight': 10, 'max_depth': 4, 'n_estimators': 84}. Best is trial 0 with value: 0.972027972027972.
[I 2026-08-25 06:10:26,991] Trial 3 finished with value: 0.9790209790209791 and parameters: {'min_child_weight': 4, 'max_depth': 4, 'n_estimators': 141}. Best is trial 3 with value: 0.9790209790209791.
[I 2026-08-25 06:10:27,045] Trial 4 finished with value: 0.965034965034965 and parameters: {'min_child_weight': 8, 'max_depth': 10, 'n_estimators': 184}. Best is trial 3 with value: 0.9790209790209

Mejores resultados

In [232]:
print("Best parameters:")
print(contenedor.best_params) #atributo optuna q devuelve los mejores hiperparametros

Best parameters:
{'min_child_weight': 4, 'max_depth': 4, 'n_estimators': 141}


In [233]:
print("Best accuracy:")
print(contenedor.best_value) #atributo optuna q devuelve el mejor puntaje (accuracy)

Best accuracy:
0.9790209790209791


In [234]:
optuna.visualization.plot_optimization_history(contenedor) #Muestra la evolucion del entrenamiento

In [235]:
optuna.visualization.plot_param_importances(contenedor)
#Influencia/importancia de lso hiperparametros
#Util para saber en hiperparametro vale la pena invertir mas tiempo refinando

In [236]:
param_grid = {
    "min_child_weight": [1, 3, 5, 7, 10],
    "max_depth": [1, 2, 4, 6, 8, 10],
    "n_estimators": [50, 100, 200, 300]
}

model = XGBClassifier(eval_metric="logloss",random_state=42) #métrica de evaluación para modelos de clasificación que mide qué tan acertadas y seguras son sus predicciones de probabilidad.

grid = GridSearchCV(model,param_grid,cv=5,scoring="accuracy" )

grid.fit(X_train, y_train)

print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 1, 'min_child_weight': 1, 'n_estimators': 100}
0.9718194254445965


In [237]:
param_dist = {
    "min_child_weight": randint(1, 11),
    "max_depth": randint(1, 11),
    "n_estimators": randint(50, 301)
}

random_search = RandomizedSearchCV(model,param_distributions=param_dist, n_iter=30, cv=5, scoring="accuracy",random_state=42)

random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)

{'max_depth': 5, 'min_child_weight': 1, 'n_estimators': 253}
0.9624076607387142


# **Entrenamiento y Evaluacion con los mejores hiperparametros**

In [239]:
# Mejores hiperparámetros encontrados
best_params = contenedor.best_params
print("Mejores hiperparámetros:")
print(best_params)

Mejores hiperparámetros:
{'min_child_weight': 4, 'max_depth': 4, 'n_estimators': 141}


In [242]:
# Crear modelo con los mejores hiperparámetros
# ** significa desempaquetar un diccionario de los Mejores hiperparametros
best_model = XGBClassifier(**best_params,eval_metric="logloss")

In [243]:
# Entrenar con los datos de entrenamiento
best_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=4, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=141, n_jobs=None,
              num_parallel_tree=None, ...)

In [245]:
# Evaluar en X_test
test_score = best_model.score(X_test, y_test)
test_score

0.9790209790209791